# 02 - Entraînement du modèle

Ce notebook implémente la boucle d'entraînement du `CurlClassifier` avec une validation croisée (80/20 split) et sauvegarde du meilleur modèle.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import os
import sys

# Add src to path
sys.path.append('../src')
from utils import load_dataset
from model import CurlClassifier
from export_mobile import export_to_torchscript

## 1. Chargement et préparation des données

Nous chargeons les données et les convertissons au format Tensor PyTorch.

In [2]:
print("Loading dataset...")
X, y = load_dataset('../data')

if len(X) == 0:
    print("Error: No data found in 'data/' directory.")
else:
    # Transpose X to (N, C, L) for Conv1d: (batch, channels, seq_len)
    X = np.transpose(X, (0, 2, 1))
    
    # Normalization
    mean = X.mean(axis=(0, 2))
    std = X.std(axis=(0, 2))
    print(f"Normalization Stats:")
    print(f"Mean: {mean.tolist()}")
    print(f"Std:  {std.tolist()}")
    
    X = (X - mean[None, :, None]) / (std[None, :, None] + 1e-7)
    
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.long)
    
    dataset = TensorDataset(X_tensor, y_tensor)
    
    # Split into train and validation
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
    
    print(f"\nDataset prepared:")
    print(f"Train size: {train_size}")
    print(f"Val size: {val_size}")
    print(f"Total samples: {len(dataset)}")

Loading dataset...
Normalization Stats:
Mean: [-0.7676139916403193, 0.5601637043184527, 3.542912663921369, -0.0014074764479736581, 0.02711421065678365, 0.009322313179625601]
Std:  [3.4267641144905943, 1.6899260425355824, 7.461371655943027, 1.6466829848394424, 1.9945634180299703, 1.1331183283993016]

Dataset prepared:
Train size: 100
Val size: 25
Total samples: 125


## 2. Configuration de l'entraînement

In [3]:
epochs = 30
batch_size = 32
lr = 0.001

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

model = CurlClassifier()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

best_val_loss = float('inf')

print(f"Training configuration:")
print(f"  Epochs: {epochs}")
print(f"  Batch size: {batch_size}")
print(f"  Learning rate: {lr}")
print(f"  Train batches per epoch: {len(train_loader)}")
print(f"  Val batches per epoch: {len(val_loader)}")

Training configuration:
  Epochs: 30
  Batch size: 32
  Learning rate: 0.001
  Train batches per epoch: 4
  Val batches per epoch: 1


## 3. Boucle d'entraînement

Entraînement avec validation et sauvegarde du meilleur modèle basée sur la loss de validation.

In [4]:
print(f"Starting training for {epochs} epochs...")
for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += batch_y.size(0)
        correct += predicted.eq(batch_y).sum().item()
        
    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += batch_y.size(0)
            val_correct += predicted.eq(batch_y).sum().item()
    
    avg_val_loss = val_loss / len(val_loader)
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        os.makedirs('../models_saved', exist_ok=True)
        export_to_torchscript(model, '../models_saved/curl_classifier.pt')
        print(f"** Saved best model to models_saved/curl_classifier.pt (Val Loss: {best_val_loss:.4f})")

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{epochs}: "
              f"Loss: {train_loss/len(train_loader):.4f}, Acc: {100.*correct/total:.2f}% | "
              f"Val Loss: {avg_val_loss:.4f}, Val Acc: {100.*val_correct/val_total:.2f}%")

print(f"Training finished. Best Val Loss: {best_val_loss:.4f}")

Starting training for 30 epochs...
Model exported and optimized for mobile: ../models_saved/curl_classifier.pt
** Saved best model to models_saved/curl_classifier.pt (Val Loss: 0.5251)
Epoch 1/30: Loss: 0.6653, Acc: 45.00% | Val Loss: 0.5251, Val Acc: 92.00%
Model exported and optimized for mobile: ../models_saved/curl_classifier.pt
** Saved best model to models_saved/curl_classifier.pt (Val Loss: 0.3259)
Model exported and optimized for mobile: ../models_saved/curl_classifier.pt
** Saved best model to models_saved/curl_classifier.pt (Val Loss: 0.2612)
Model exported and optimized for mobile: ../models_saved/curl_classifier.pt
** Saved best model to models_saved/curl_classifier.pt (Val Loss: 0.1992)
Model exported and optimized for mobile: ../models_saved/curl_classifier.pt
** Saved best model to models_saved/curl_classifier.pt (Val Loss: 0.1572)
Epoch 5/30: Loss: 0.2286, Acc: 90.00% | Val Loss: 0.1572, Val Acc: 92.00%
Model exported and optimized for mobile: ../models_saved/curl_class